# Airbnb (Амстердам 2025)

## 0. Описание выбора компании и формулировка двух связанных бизнес-задач

Сервис Airbnb предоставляет возможность владельцам жилья размещать объявления о краткосрочной аренде недвижимости. Для людей, которые сдают жилье важным является понимание факторов, влияющих на коммерческую успешность объекта: уровня цены, характеристик жилья, расположения, удобств и пользовательских отзывов.

В рамках проекта рассматриваются данные по объявлениям Airbnb в Амстердаме, опубликованные в открытом наборе данных [Inside Airbnb](https://insideairbnb.com/get-the-data/#:~:text=Amsterdam%2C%20North%20Holland%2C%20The%20Netherlands).

Основная цель проекта заключается в прогнозировании коммерческой успешности объявлений Airbnb. При этом в качестве критерия успешности используется не абсолютная годовая выручка, поскольку высокий доход часто обусловлен исключительно высокой стоимостью аренды. Вместо этого рассматривается показатель относительной эффективности, отражающий способность объявления генерировать доход относительно своей цены за ночь.

### Бизнес-задача 1. Оценка успешности объявления по его характеристикам

При создании нового объявления владелец жилья должен определить цену, заполнить описание объекта, указать доступные удобства и условия бронирования. Однако на этом этапе отсутствует информация о будущих бронированиях и фактической доходности объекта.

Разрабатываемая модель позволяет прогнозировать коммерческую успешность объявления на основе его структурированных характеристик: расположения, типа жилья, вместимости, стоимости проживания, удобств и других параметров. Такой прогноз может использоваться до публикации объявления или при его обновлении для оценки потенциальной эффективности объекта.

В рамках бизнес-процесса модель может выступать инструментом поддержки принятия решений. Полученный прогноз помогает владельцу определить, насколько конкурентоспособным является объявление относительно аналогичных объектов, а также выявить характеристики, которые могут быть улучшены для повышения коммерческой эффективности.

В рамках данной задачи модели необходимо предсказать значение целевой переменной `high_revenue_efficiency` по структурированным характеристикам объявления. В качестве признаков используются район размещения, тип жилья, вместимость, цена, набор удобств, а также ограничения и условия бронирования. Для решения задачи будет применяется полносвязная нейронная сеть (Multilayer Perceptron, MLP).

### Бизнес-задача 2. Оценка успешности объявления по отзывам гостей

Отзывы гостей содержат информацию о реальном опыте проживания и отражают аспекты, которые не всегда представлены в структурированных характеристиках объявления. К таким аспектам относятся качество коммуникации с хозяином, чистота жилья, удобство расположения, соответствие ожиданиям гостей и другие факторы, влияющие на удовлетворённость клиентов.

Разрабатываемая модель позволяет прогнозировать коммерческую успешность объявления на основе текстов отзывов. Это даёт возможность определить, насколько пользовательский опыт связан с эффективностью объекта и какие темы чаще встречаются в отзывах успешных объявлений.

В бизнес-процессе такое решение может использоваться для анализа обратной связи от гостей и выявления направлений для улучшения качества сервиса. Результаты модели помогают владельцам понять, какие аспекты проживания оказывают наибольшее влияние на успешность объекта и требуют особого внимания.

В данной задаче модели будет предсказывать ту же целевую переменную `high_revenue_efficiency` на основе текстов отзывов гостей.

В дальнейшем исследование может быть расширено за счет включения данных из других городов и стран. Это позволит построить более универсальную модель, способную выявлять закономерности рынка краткосрочной аренды независимо от конкретной локации. Также увеличение объёма данных может повысить обобщающую способность модели и устойчивость результатов. Однако расширение исследования на множество городов и стран приведет к существенному увеличению объема данных, что потребует больших вычислительных ресурсов, более длительного обучения моделей и дополнительных затрат на хранение и обработку информации.

## 1. Загрузка и первичный обзор данных

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

In [ ]:
calendar = pd.read_csv("calendar.csv.gz")
listings = pd.read_csv("listings.csv.gz")
reviews = pd.read_csv("reviews.csv.gz")

In [ ]:
print("calendar:", calendar.shape)
print("listings:", listings.shape)
print("reviews:", reviews.shape)

calendar: (3825200, 7)
listings: (10480, 79)
reviews: (501084, 6)


### 1.1. Описание таблицы `Calendar`

In [ ]:
calendar.sample(5, random_state=42)

,listing_id,date,available,price,adjusted_price,minimum_nights,maximum_nights
1901130,926614845594068398,2026-04-09,f,NaN,NaN,3,365
2475759,1161468388487708665,2026-08-06,f,NaN,NaN,1,21
1291659,645415424366456737,2026-06-27,f,NaN,NaN,3,1001
668788,35285186,2025-12-28,f,NaN,NaN,3,1125
3282933,1492075900250054449,2026-01-12,t,NaN,NaN,30,365


In [ ]:
calendar.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3825200 entries, 0 to 3825199
Data columns (total 7 columns):
 #   Column          Dtype  
---  ------          -----  
 0   listing_id      int64  
 1   date            object 
 2   available       object 
 3   price           float64
 4   adjusted_price  float64
 5   minimum_nights  int64  
 6   maximum_nights  int64  
dtypes: float64(2), int64(3), object(2)
memory usage: 204.3+ MB


Датасет Calendar содержит календарную информацию по объявлениям Airbnb и отражает доступность объектов размещения на каждую дату в течение рассматриваемого периода.

В наборе данных содержится 3 825 200 наблюдений и 7 признаков. Каждая запись соответствует конкретному объявлению в определенную дату.

Признаки датасета:

* `listing_id` – уникальный идентификатор объявления
* `date` – дата наблюдения
* `available` – индикатор доступности объекта для бронирования на указанную дату
* `price` – стоимость проживания на указанную дату
* `adjusted_price` – скорректированная стоимость проживания;
* `minimum_nights` – минимальное количество ночей для бронирования;
* `maximum_nights` – максимальное количество ночей для бронирования.

### 1.2. Описание таблицы `Listings`

In [ ]:
listings.sample(5, random_state=42)

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,host_url,host_name,host_since,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_thumbnail_url,host_picture_url,host_neighbourhood,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms,beds,amenities,price,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,calendar_updated,has_availability,availability_30,availability_60,availability_90,availability_365,calendar_last_scraped,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
10125,1455656325647057805,https://www.airbnb.com/rooms/1455656325647057805,20250911031321,2025-09-11,city scrape,Loft apartment in former school,"Fully renovated loft style apartment in a former school building, dating from 1888. Perfectly suited for a family wi...",NaN,https://a0.muscache.com/pictures/hosting/Hosting-U3RheVN1cHBseUxpc3Rpbmc6MTQ1NTY1NjMyNTY0NzA1NzgwNQ==/original/1da91...,23725517,https://www.airbnb.com/users/show/23725517,Rein,2014-11-14,NaN,NaN,NaN,NaN,NaN,f,https://a0.muscache.com/im/pictures/user/User/original/0c30799a-4b41-4b2f-91e9-b4f33bf0c554.jpeg?aki_policy=profile_...,https://a0.muscache.com/im/pictures/user/User/original/0c30799a-4b41-4b2f-91e9-b4f33bf0c554.jpeg?aki_policy=profile_...,NaN,1.0,1.0,"['email', 'phone']",t,t,NaN,Oud-Oost,NaN,52.358980,4.922760,Entire loft,Entire home/apt,5,2.0,2 baths,3.0,3.0,"[""Outdoor dining area"", ""Carbon monoxide alarm"", ""Smoke alarm"", ""TV"", ""Wifi"", ""First aid kit"", ""Paid parking on prem...",$620.00,7,30,7.0,7.0,30.0,30.0,7.0,30.0,NaN,t,0,0,0,67,2025-09-11,0,0,0,0,0,0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0363 184D A1DF F940 E474,f,1,1,0,0,NaN
1082,8798384,https://www.airbnb.com/rooms/8798384,20250911031321,2025-09-11,city scrape,Family appartement with garden near centrum.,"Apartment for a family or (adult) couple only in a quiet green child-friendly neighborhood, the popular Watergraafsm...","The appartment is located in a quiet and charming neighbourhood, the Watergraafsmeer part of the East side of Amster...",https://a0.muscache.com/pictures/miso/Hosting-8798384/original/13299983-ba62-4c79-84a9-bed6f511f6a8.jpeg,46104716,https://www.airbnb.com/users/show/46104716,Hadewijch,2015-10-08,"Amsterdam, Netherlands",NaN,within a day,100%,25%,f,https://a0.muscache.com/im/pictures/user/ada0b6e5-6fe5-4dc0-bdb1-ca4c22ac4699.jpg?aki_policy=profile_small,https://a0.muscache.com/im/pictures/user/ada0b6e5-6fe5-4dc0-bdb1-ca4c22ac4699.jpg?aki_policy=profile_x_medium,NaN,1.0,1.0,"['email', 'phone']",t,t,"Amsterdam, NH, Netherlands",Watergraafsmeer,NaN,52.351100,4.940850,Entire rental unit,Entire home/apt,4,1.5,1.5 baths,3.0,3.0,"[""Coffee"", ""Portable fans"", ""Toaster"", ""Free dryer \u2013 In unit"", ""Books and reading material"", ""Hangers"", ""Hot wa...",$181.00,3,30,3.0,3.0,30.0,30.0,3.0,30.0,NaN,t,2,2,3,131,2025-09-11,25,3,0,3,5,23,4163.0,2016-03-30,2025-08-08,4.88,4.92,4.80,4.88,4.96,4.92,4.76,0363 6BA3 D9EE 979C F004,f,1,1,0,0,0.22
713,4984717,https://www.airbnb.com/rooms/4984717,20250911031321,2025-09-11,city scrape,The Blue room,Welcome in our 17th centery monumental ho

In [ ]:
listings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10480 entries, 0 to 10479
Data columns (total 79 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            10480 non-null  int64  
 1   listing_url                                   10480 non-null  object 
 2   scrape_id                                     10480 non-null  int64  
 3   last_scraped                                  10480 non-null  object 
 4   source                                        10480 non-null  object 
 5   name                                          10480 non-null  object 
 6   description                                   10132 non-null  object 
 7   neighborhood_overview                         5192 non-null   object 
 8   picture_url                                   10480 non-null  object 
 9   host_id                                       10480 non-null 

Датасет Listings содержит подробную информацию об объявлениях Airbnb в Амстердаме и является основным источником признаков для решения задачи классификации. Каждая запись соответствует одному объекту размещения.

Набор данных включает 10 480 объявлений и 79 признаков, характеризующих как само жилье, так и его владельца, условия бронирования, стоимость проживания и уровень доступности объекта.

Признаки датасета можно разделить на несколько групп:

* Общая информация об объявлении – название объекта, описание, тип жилья, тип комнаты, ссылки на объявление и фотографии
* Географические характеристики – район размещения, координаты объекта (широта и долгота)
* Характеристики жилья – вместимость, количество спален, кроватей и ванных комнат
* Ценовые характеристики – стоимость проживания и ограничения по минимальному и максимальному сроку бронирования
* Информация о хосте – дата регистрации, местоположение, статус Superhost, показатели отклика и подтверждения бронирований
* Удобства – перечень доступных сервисов и удобств, предоставляемых гостям.
* Показатели доступности – количество доступных дней для бронирования на различных временных горизонтах

### 1.3. Описание таблицы `Reviews`

In [ ]:
reviews.sample(5, random_state=42)

,listing_id,id,date,reviewer_id,reviewer_name,comments
248665,21009564,1423827034128714224,2025-05-18,51237368,Tom,"A lovely place, perfect location with a real easy feel to it. Great for a chill in the city. Steep stairs if you str..."
104641,4640869,827447009300697548,2023-02-15,205934491,Levin,Marielle and her place were so nice! It is a very cute apartment and Marielle was always responding fast if we had a...
209901,15963303,173452274,2017-07-23,50958060,Franck,Try and see... it's worth it! The boat is amazing and the terrace THE place to relax after a busy day. Very close to...
171470,11161861,983895883542734324,2023-09-19,78326522,Alejandra,"Excelente ubicación, muy cerca del tranvía, metro y estaciones de bus, la zona tiene restaurantes, bares, tiendas, d..."
375676,43282992,886023407843432164,2023-05-07,501511088,Prashantha,The stay at Lisa's place was very nice. it's a 10 minute walk from the metro. You will find some Turkey restaurants...


In [ ]:
reviews.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 501084 entries, 0 to 501083
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   listing_id     501084 non-null  int64 
 1   id             501084 non-null  int64 
 2   date           501084 non-null  object
 3   reviewer_id    501084 non-null  int64 
 4   reviewer_name  501083 non-null  object
 5   comments       501053 non-null  object
dtypes: int64(3), object(3)
memory usage: 22.9+ MB


Датасет Reviews содержит текстовые отзывы гостей Airbnb и используется для решения задачи классификации на основе неструктурированных данных. Каждая запись соответствует одному отзыву, оставленному пользователем после проживания в объекте размещения.

Набор данных включает 501 084 отзыва и 6 признаков:

* `listing_id` – идентификатор объявления, к которому относится отзыв
* `id` – уникальный идентификатор отзыва
* `date` – дата публикации отзыва
* `reviewer_id` – идентификатор автора отзыва
* `reviewer_name` – имя автора отзыва
* `comments` – текст отзыва

Наибольшую ценность для исследования представляет признак `comments`, содержащий свободный текст на различных языках. В отзывах гости описывают свой опыт проживания, качество жилья, удобство расположения, взаимодействие с хозяином, уровень чистоты и другие аспекты, которые могут быть связаны с коммерческой успешностью объекта.

## 2. Предобработка данных

### 2.1. Предобработка `Сalendar`

In [ ]:
calendar.sample(5, random_state=42)

,listing_id,date,available,price,adjusted_price,minimum_nights,maximum_nights
1901130,926614845594068398,2026-04-09,f,NaN,NaN,3,365
2475759,1161468388487708665,2026-08-06,f,NaN,NaN,1,21
1291659,645415424366456737,2026-06-27,f,NaN,NaN,3,1001
668788,35285186,2025-12-28,f,NaN,NaN,3,1125
3282933,1492075900250054449,2026-01-12,t,NaN,NaN,30,365


Проверим наличие пропущенных значений

In [ ]:
calendar.isna().sum()

,0
listing_id,0
date,0
available,0
price,3825200
adjusted_price,3825200
minimum_nights,0
maximum_nights,0


Признаки `price` и `adjusted_price` полностью состоят из пропусков. Поскольку информация о стоимости присутствует в датасете Listings, удалим данные столбцы

In [ ]:
calendar = calendar.drop(columns=["price", "adjusted_price"])

Исследуем распределение признака доступности объектов

In [ ]:
calendar["available"].value_counts(dropna=False)

,count
available,
f,2839532
t,985668


Большая часть записей соответствует недоступным для бронирования датам (`f`). Доступные даты составляют меньшую долю наблюдений.

Для упрощения дальнейших вычислений преобразуем категориальный признак доступности в бинарный формат.

In [ ]:
calendar["is_available"] = (calendar["available"] == "t").astype(int)
calendar["is_unavailable"] = (calendar["available"] == "f").astype(int)

Для работы с временными признаками преобразуем столбец даты в формат datetime

In [ ]:
calendar["date"] = pd.to_datetime(calendar["date"])

In [ ]:
calendar[["date", "available", "is_available", "is_unavailable"]].sample(5, random_state=42)

,date,available,is_available,is_unavailable
1901130,2026-04-09,f,0,1
2475759,2026-08-06,f,0,1
1291659,2026-06-27,f,0,1
668788,2025-12-28,f,0,1
3282933,2026-01-12,t,1,0


Из даты извлечем дополнительные временные признаки, которые могут влиять на спрос и загрузку объектов.

In [ ]:
calendar["month"] = calendar["date"].dt.month
calendar["day_of_week"] = calendar["date"].dt.dayofweek
calendar["is_weekend"] = calendar["day_of_week"].isin([5, 6]).astype(int)

### 2.2. Предобработка `Listings`

В Listings есть много служебных колонок: ссылки, id и пустые поля. Для модели они не несут большого смысла, поэтому дальше оставим только признаки, которые описывают жилье, владельца жилья, район, цену и текст объявления

In [ ]:
listings.sample(5, random_state=42)

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,host_url,host_name,host_since,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_thumbnail_url,host_picture_url,host_neighbourhood,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms,beds,amenities,price,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,calendar_updated,has_availability,availability_30,availability_60,availability_90,availability_365,calendar_last_scraped,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
10125,1455656325647057805,https://www.airbnb.com/rooms/1455656325647057805,20250911031321,2025-09-11,city scrape,Loft apartment in former school,"Fully renovated loft style apartment in a former school building, dating from 1888. Perfectly suited for a family wi...",NaN,https://a0.muscache.com/pictures/hosting/Hosting-U3RheVN1cHBseUxpc3Rpbmc6MTQ1NTY1NjMyNTY0NzA1NzgwNQ==/original/1da91...,23725517,https://www.airbnb.com/users/show/23725517,Rein,2014-11-14,NaN,NaN,NaN,NaN,NaN,f,https://a0.muscache.com/im/pictures/user/User/original/0c30799a-4b41-4b2f-91e9-b4f33bf0c554.jpeg?aki_policy=profile_...,https://a0.muscache.com/im/pictures/user/User/original/0c30799a-4b41-4b2f-91e9-b4f33bf0c554.jpeg?aki_policy=profile_...,NaN,1.0,1.0,"['email', 'phone']",t,t,NaN,Oud-Oost,NaN,52.358980,4.922760,Entire loft,Entire home/apt,5,2.0,2 baths,3.0,3.0,"[""Outdoor dining area"", ""Carbon monoxide alarm"", ""Smoke alarm"", ""TV"", ""Wifi"", ""First aid kit"", ""Paid parking on prem...",$620.00,7,30,7.0,7.0,30.0,30.0,7.0,30.0,NaN,t,0,0,0,67,2025-09-11,0,0,0,0,0,0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0363 184D A1DF F940 E474,f,1,1,0,0,NaN
1082,8798384,https://www.airbnb.com/rooms/8798384,20250911031321,2025-09-11,city scrape,Family appartement with garden near centrum.,"Apartment for a family or (adult) couple only in a quiet green child-friendly neighborhood, the popular Watergraafsm...","The appartment is located in a quiet and charming neighbourhood, the Watergraafsmeer part of the East side of Amster...",https://a0.muscache.com/pictures/miso/Hosting-8798384/original/13299983-ba62-4c79-84a9-bed6f511f6a8.jpeg,46104716,https://www.airbnb.com/users/show/46104716,Hadewijch,2015-10-08,"Amsterdam, Netherlands",NaN,within a day,100%,25%,f,https://a0.muscache.com/im/pictures/user/ada0b6e5-6fe5-4dc0-bdb1-ca4c22ac4699.jpg?aki_policy=profile_small,https://a0.muscache.com/im/pictures/user/ada0b6e5-6fe5-4dc0-bdb1-ca4c22ac4699.jpg?aki_policy=profile_x_medium,NaN,1.0,1.0,"['email', 'phone']",t,t,"Amsterdam, NH, Netherlands",Watergraafsmeer,NaN,52.351100,4.940850,Entire rental unit,Entire home/apt,4,1.5,1.5 baths,3.0,3.0,"[""Coffee"", ""Portable fans"", ""Toaster"", ""Free dryer \u2013 In unit"", ""Books and reading material"", ""Hangers"", ""Hot wa...",$181.00,3,30,3.0,3.0,30.0,30.0,3.0,30.0,NaN,t,2,2,3,131,2025-09-11,25,3,0,3,5,23,4163.0,2016-03-30,2025-08-08,4.88,4.92,4.80,4.88,4.96,4.92,4.76,0363 6BA3 D9EE 979C F004,f,1,1,0,0,0.22
713,4984717,https://www.airbnb.com/rooms/4984717,20250911031321,2025-09-11,city scrape,The Blue room,Welcome in our 17th centery monumental ho

In [ ]:
url_cols = [col for col in listings.columns if "url" in col]

service_cols = [
    "scrape_id",
    "last_scraped",
    "source",
    "calendar_updated",
    "calendar_last_scraped",
    "host_thumbnail_url",
    "host_picture_url",
    "picture_url",
]

drop_now = url_cols + service_cols
drop_now

['listing_url',
 'picture_url',
 'host_url',
 'host_thumbnail_url',
 'host_picture_url',
 'scrape_id',
 'last_scraped',
 'source',
 'calendar_updated',
 'calendar_last_scraped',
 'host_thumbnail_url',
 'host_picture_url',
 'picture_url']

In [ ]:
listings = listings.drop(columns=drop_now)

In [ ]:
listings[["price", "host_response_rate", "host_acceptance_rate"]].sample(5, random_state=42)

,price,host_response_rate,host_acceptance_rate
10125,$620.00,NaN,NaN
1082,$181.00,100%,25%
713,$169.00,90%,89%
1683,$159.00,90%,59%
1111,$134.00,100%,100%


Некоторые числовые признаки хранятся в строковом формате. В частности, цена записана с символом валюты, а показатели отклика и принятия заявок хоста представлены в процентах. Перед обучением модели приведем их к числовому виду

In [ ]:
listings["price"] = (
    listings["price"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .replace("nan", np.nan)
    .astype(float)
)

In [ ]:
for col in ["host_response_rate", "host_acceptance_rate"]:
    listings[col] = (
        listings[col]
        .astype(str)
        .str.replace("%", "", regex=False)
        .replace("nan", np.nan)
        .astype(float) / 100
    )

In [ ]:
listings[["price", "host_response_rate", "host_acceptance_rate"]].describe()

,price,host_response_rate,host_acceptance_rate
count,5874.000000,6683.000000,8050.000000
mean,336.785155,0.918074,0.715062
std,1985.661882,0.210643,0.323423
min,35.000000,0.000000,0.000000
25%,161.000000,1.000000,0.500000
50%,222.000000,1.000000,0.830000
75%,314.000000,1.000000,1.000000
max,80018.000000,1.000000,1.000000


Пропуски пока не заполняем, сначала соберем все нужные признаки, а уже перед обучением обработаем NaN единообразно

Часть признаков записана как `t` или `f`. Для модели удобнее перевести их в 1/0

In [ ]:
bool_cols = [
    "host_is_superhost",
    "host_has_profile_pic",
    "host_identity_verified",
    "instant_bookable",
]

In [ ]:
for col in bool_cols:
    listings[col] = listings[col].map({"t": 1, "f": 0})
    listings[col] = listings[col].fillna(listings[col].mode()[0])

In [ ]:
listings[bool_cols].isna().sum()

,0
host_is_superhost,0
host_has_profile_pic,0
host_identity_verified,0
instant_bookable,0


Признак `has_availability` не содержит существенной информации, чтобы различать объявления, поэтому колонку удалим.

In [ ]:
listings = listings.drop(columns=["has_availability"])

Вместо даты регистрации хоста создадим более интерпретируемый признак – количество лет присутствия на платформе Airbnb

In [ ]:
listings["host_since"] = pd.to_datetime(listings["host_since"], errors="coerce")

scrape_date = pd.to_datetime("2025-09-11")

listings["host_years"] = (
    scrape_date - listings["host_since"]
).dt.days / 365

In [ ]:
listings["host_years"] = listings["host_years"].fillna(
    listings["host_years"].median()
)

listings["host_years"].describe()

,host_years
count,10480.000000
mean,9.120041
std,3.401963
min,0.016438
25%,7.360274
50%,9.931507
75%,11.518493
max,17.095890


Поскольку признак `host_years` характеризует стаж владельца жилья на платформе и имеет разброс от недавно зарегистрированных пользователей до владельцев с многолетним опытом, для заполнения пропусков была выбрана медиана как наиболее устойчивая оценка типичного значения признака

Текстовые поля содержат полезную информацию об объекте и его владельце. Для корректной обработки заменим пропущенные значения пустой строкой

In [ ]:
text_cols = ["name", "description", "neighborhood_overview", "host_about"]

for col in text_cols:
    listings[col] = listings[col].fillna("")

Вместо использования полного текста на данном этапе создадим простые количественные характеристики, отражающие объем информации, представленной в объявлении

In [ ]:
listings["name_len"] = listings["name"].str.len()
listings["description_len"] = listings["description"].str.len()
listings["neighborhood_overview_len"] = listings["neighborhood_overview"].str.len()
listings["host_about_len"] = listings["host_about"].str.len()

listings["has_description"] = (listings["description_len"] > 0).astype(int)
listings["has_neighborhood_overview"] = (
    listings["neighborhood_overview_len"] > 0
).astype(int)
listings["has_host_about"] = (listings["host_about_len"] > 0).astype(int)

Для каждого текстового поля рассчитана длина текста, а также признаки наличия описания

Количество и состав удобств являются важными характеристиками объекта. Сформируем признаки, отражающие как общий объем удобств, так и наличие наиболее востребованных сервисов

In [ ]:
listings["amenities_count"] = listings["amenities"].str.count(",") + 1
listings.loc[listings["amenities"] == "[]", "amenities_count"] = 0

Рассчитано общее количество удобств, указанных в объявлении. Данный показатель может служить косвенной характеристикой качества и оснащенности жилья

In [ ]:
amenities_text = listings["amenities"].str.lower()

listings["has_wifi"] = amenities_text.str.contains("wifi", regex=False).astype(int)
listings["has_kitchen"] = amenities_text.str.contains("kitchen", regex=False).astype(int)
listings["has_washer"]= amenities_text.str.contains("washer", regex=False).astype(int)
listings["has_air_conditioning"] =amenities_text.str.contains("air conditioning", regex=False).astype(int)
listings["has_parking"] = amenities_text.str.contains("parking",regex=False).astype(int)
listings["has_workspace"] = amenities_text.str.contains("workspace",regex=False).astype(int)

Дополнительно сформированы бинарные признаки наличия наиболее популярных удобств: Wi-Fi, кухни, стиральной машины, кондиционера, парковки и рабочего места

In [ ]:
listings[
    [
        "host_years",
        "description_len",
        "neighborhood_overview_len",
        "host_about_len",
        "amenities_count",
        "has_wifi",
        "has_kitchen",
        "has_parking",
    ]
].sample(5, random_state=42)

,host_years,description_len,neighborhood_overview_len,host_about_len,amenities_count,has_wifi,has_kitchen,has_parking
10125,10.832877,416,0,0,13,1,1,1
1082,9.934247,481,343,0,51,1,1,1
713,11.600000,433,84,374,32,1,0,0
1683,12.726027,465,195,78,25,1,1,1
1111,12.391781,410,291,251,24,1,1,1


### 2.3. Предобработка `Reviews`

In [ ]:
reviews.sample(5, random_state=42)

,listing_id,id,date,reviewer_id,reviewer_name,comments
248665,21009564,1423827034128714224,2025-05-18,51237368,Tom,"A lovely place, perfect location with a real easy feel to it. Great for a chill in the city. Steep stairs if you str..."
104641,4640869,827447009300697548,2023-02-15,205934491,Levin,Marielle and her place were so nice! It is a very cute apartment and Marielle was always responding fast if we had a...
209901,15963303,173452274,2017-07-23,50958060,Franck,Try and see... it's worth it! The boat is amazing and the terrace THE place to relax after a busy day. Very close to...
171470,11161861,983895883542734324,2023-09-19,78326522,Alejandra,"Excelente ubicación, muy cerca del tranvía, metro y estaciones de bus, la zona tiene restaurantes, bares, tiendas, d..."
375676,43282992,886023407843432164,2023-05-07,501511088,Prashantha,The stay at Lisa's place was very nice. it's a 10 minute walk from the metro. You will find some Turkey restaurants...


Проверим качество данных и наличие пропущенных значений

In [ ]:
reviews.isna().mean().sort_values(ascending=False)

,0
comments,0.000062
reviewer_name,0.000002
id,0.000000
listing_id,0.000000
reviewer_id,0.000000
date,0.000000


Отсутствующие значения встречаются только в признаках comments и reviewer_name, причем их доля пренебрежимо мала

Для последующей работы с временной информацией приведем дату публикации отзыва к формату datetime

In [ ]:
reviews["date"] = pd.to_datetime(reviews["date"], errors="coerce")

Основным источником информации в данном датасете является текст отзыва. Для корректной обработки заменим редкие пропуски пустой строкой и приведем данные к строковому типу

In [ ]:
reviews["comments"] = reviews["comments"].fillna("").astype(str)

Помимо самих текстов отзывов создадим дополнительные количественные характеристики, отражающие объем пользовательского комментария

In [ ]:
reviews["comment_len"] = reviews["comments"].str.len()
reviews["comment_word_count"] = reviews["comments"].str.split().str.len()


In [ ]:
reviews[["date", "comments", "comment_len", "comment_word_count"]].sample(5, random_state=42)

,date,comments,comment_len,comment_word_count
248665,2025-05-18,"A lovely place, perfect location with a real easy feel to it. Great for a chill in the city. Steep stairs if you str...",172,33
104641,2023-02-15,Marielle and her place were so nice! It is a very cute apartment and Marielle was always responding fast if we had a...,153,28
209901,2017-07-23,Try and see... it's worth it! The boat is amazing and the terrace THE place to relax after a busy day. Very close to...,224,43
171470,2023-09-19,"Excelente ubicación, muy cerca del tranvía, metro y estaciones de bus, la zona tiene restaurantes, bares, tiendas, d...",269,43
375676,2023-05-07,The stay at Lisa's place was very nice. it's a 10 minute walk from the metro. You will find some Turkey restaurants...,438,71
